# 07 — Improve the graph branch without replacing DINO

This notebook keeps the existing baselines untouched and tests targeted changes motivated by the current results.

Changes tested here:

1. **Frozen local-patch baseline** — local DINO patch matching without any GNN.
2. **DINO-preserving GATv2 encoder** — GAT predicts a residual correction to each original 384-D DINO patch rather than replacing it with a 256-D representation.
3. **Patch-to-patch readout** — explicit max-mean local matching instead of averaging all patches before cosine similarity.
4. **Non-zero score residual initialization** — final score residual `alpha` starts at `0.05` instead of `0`, so the graph branch receives gradient immediately.

The first improved run keeps `top_k=10` unchanged so graph density is not changed at the same time. Tune `top_k` and thresholds only after this architecture is evaluated on validation.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")
BRANCH_NAME = "experiment/max-mean-readout"

if not REPO_DIR.exists():
    !git clone -b "$BRANCH_NAME" "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" fetch
    !git -C "$REPO_DIR" checkout "$BRANCH_NAME"
    !git -C "$REPO_DIR" pull origin "$BRANCH_NAME"

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


In [ ]:
import json
import math
import torch
import torch.nn.functional as F
from torch import nn

from cross_image_glot.config import DEFAULT_PATHS
from cross_image_glot.models import (
    PatchGATv2Encoder,
    CrossImageGraphMatcher,
    BaselinePreservingResidualMatcher,
)
from cross_image_glot.storage import restore_feature_splits, atomic_json_save
from cross_image_glot.data import MiniImageNetFeatureDataset, FewShotFeatureEpisodeDataset
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder
from cross_image_glot.baselines import frozen_baseline_episode
from cross_image_glot.training import (
    evaluate_residual_dataset,
    evaluate_residual_feature_episode,
    load_training_checkpoint,
    make_checkpoint,
    save_checkpoint_atomic,
    save_history,
    train_residual_epoch,
)

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


## Experimental model components

These are defined inline so this notebook runs without modifying your existing `models.py`. The same code is also provided separately as `experimental_models.py`; if the experiment works, add that file under `src/cross_image_glot/`.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import torch
import torch.nn.functional as F
from torch import nn


@dataclass
class PatchMatchReadoutOutput:
    """One score per candidate graph."""
    scores: torch.Tensor
    raw_similarities: torch.Tensor


class DinoResidualGraphEncoder(nn.Module):
    """
    Preserve the original frozen DINOv2 patch representation and let a graph
    encoder learn only a residual correction.

    graph.x[:, :dino_dim] must contain the original DINOv2 patch token.

        base_encoder: [nodes, input_dim] -> [nodes, base_output_dim]
        delta head:   [nodes, base_output_dim] -> [nodes, dino_dim]
        output:       DINO + beta * delta

    The correction is norm-matched to the original DINO token so beta has a
    stable interpretation. beta=0.05 starts at roughly a 5% correction.
    """

    def __init__(
        self,
        base_encoder: nn.Module,
        base_output_dim: int,
        dino_dim: int = 384,
        initial_patch_residual_scale: float = 0.05,
        norm_match_delta: bool = True,
    ) -> None:
        super().__init__()
        self.base_encoder = base_encoder
        self.base_output_dim = base_output_dim
        self.dino_dim = dino_dim
        self.hidden_dim = dino_dim
        self.norm_match_delta = norm_match_delta

        self.delta_projection = nn.Linear(base_output_dim, dino_dim)
        self.patch_residual_scale = nn.Parameter(
            torch.tensor(float(initial_patch_residual_scale), dtype=torch.float32)
        )

    def forward(self, graph):
        if graph.x.shape[-1] < self.dino_dim:
            raise ValueError(
                f"graph.x has {graph.x.shape[-1]} features, but dino_dim={self.dino_dim}."
            )

        original_dino = graph.x[:, : self.dino_dim].to(torch.float32)
        graph_hidden = self.base_encoder(graph)
        delta = self.delta_projection(graph_hidden)

        if self.norm_match_delta:
            delta = F.normalize(delta, p=2, dim=-1)
            original_norm = (
                original_dino.norm(p=2, dim=-1, keepdim=True)
                .detach()
                .clamp_min(1e-6)
            )
            delta = delta * original_norm

        return original_dino + self.patch_residual_scale * delta


class MaxMeanPatchReadout(nn.Module):
    """
    Patch-to-patch scoring instead of mean-pooling all patches first.

    Default candidate score:

        mean over support images j [
            mean over query patches i [
                max over support patches p cosine(q_i, s_{j,p})
            ]
        ]

    This preserves local correspondences and gives each support image equal
    weight. `classwide_max` instead matches each query patch against all support
    patches of the candidate class jointly.
    """

    def __init__(
        self,
        temperature: float = 0.1,
        support_reduction: str = "mean_image",
    ) -> None:
        super().__init__()
        if temperature <= 0:
            raise ValueError("temperature must be positive.")
        if support_reduction not in {"mean_image", "classwide_max"}:
            raise ValueError(
                "support_reduction must be 'mean_image' or 'classwide_max'."
            )
        self.temperature = float(temperature)
        self.support_reduction = support_reduction

    def _single_graph_score(self, nodes, image_ids):
        query = nodes[image_ids == 0]
        if query.numel() == 0:
            raise ValueError("Candidate graph has no query patches.")
        query = F.normalize(query, p=2, dim=-1)

        support_ids = torch.unique(image_ids[image_ids > 0], sorted=True)
        if support_ids.numel() == 0:
            raise ValueError("Candidate graph has no support patches.")

        if self.support_reduction == "classwide_max":
            support = F.normalize(nodes[image_ids > 0], p=2, dim=-1)
            similarity = query @ support.T
            return similarity.max(dim=1).values.mean()

        per_image_scores = []
        for support_id in support_ids:
            support = F.normalize(nodes[image_ids == support_id], p=2, dim=-1)
            similarity = query @ support.T
            per_image_scores.append(similarity.max(dim=1).values.mean())
        return torch.stack(per_image_scores).mean()

    def forward(self, refined_nodes: torch.Tensor, graph_batch):
        if not hasattr(graph_batch, "image_id"):
            raise AttributeError("graph_batch must contain image_id metadata.")

        image_ids = graph_batch.image_id.reshape(-1).long()
        if hasattr(graph_batch, "batch"):
            graph_ids = graph_batch.batch.reshape(-1).long()
            num_graphs = int(graph_batch.num_graphs)
        else:
            graph_ids = torch.zeros(
                refined_nodes.shape[0], dtype=torch.long, device=refined_nodes.device
            )
            num_graphs = 1

        raw_scores = []
        for graph_id in range(num_graphs):
            mask = graph_ids == graph_id
            raw_scores.append(
                self._single_graph_score(refined_nodes[mask], image_ids[mask])
            )

        raw_scores = torch.stack(raw_scores)
        return PatchMatchReadoutOutput(
            scores=raw_scores / self.temperature,
            raw_similarities=raw_scores,
        )


@torch.inference_mode()
def frozen_patch_match_episode(
    episode: dict,
    device: torch.device,
    temperature: float = 0.1,
    support_reduction: str = "mean_image",
) -> tuple[torch.Tensor, torch.Tensor]:
    """Non-GNN local DINO patch-matching baseline."""
    support = episode["support_patches"].to(device=device, dtype=torch.float32)
    query = episode["query_patches"].to(device=device, dtype=torch.float32)
    labels = episode["query_labels"].to(device=device, dtype=torch.long)

    n_way, k_shot, _, _ = support.shape
    queries_per_class = query.shape[1]
    support = F.normalize(support, p=2, dim=-1)
    query = F.normalize(query, p=2, dim=-1)

    logits_rows = []
    targets = []

    for query_class_position in range(n_way):
        for query_position in range(queries_per_class):
            q = query[query_class_position, query_position]
            candidate_scores = []

            for candidate_id in range(n_way):
                candidate_support = support[candidate_id]

                if support_reduction == "classwide_max":
                    flat_support = candidate_support.reshape(
                        -1, candidate_support.shape[-1]
                    )
                    similarity = q @ flat_support.T
                    raw_score = similarity.max(dim=1).values.mean()
                elif support_reduction == "mean_image":
                    per_image_scores = []
                    for support_index in range(k_shot):
                        similarity = q @ candidate_support[support_index].T
                        per_image_scores.append(
                            similarity.max(dim=1).values.mean()
                        )
                    raw_score = torch.stack(per_image_scores).mean()
                else:
                    raise ValueError(
                        "support_reduction must be 'mean_image' or 'classwide_max'."
                    )

                candidate_scores.append(raw_score / temperature)

            logits_rows.append(torch.stack(candidate_scores))
            targets.append(labels[query_class_position, query_position])

    return torch.stack(logits_rows), torch.stack(targets)


In [ ]:
run_config = {
    "experiment_name": "residual_gatv2_dino_patchmatch_5way5shot",
    "n_way": 5,
    "k_shot": 5,
    "input_dim": 387,
    "dino_dim": 384,
    "hidden_dim": 256,
    "num_layers": 2,
    "attention_heads": 4,
    "edge_dim": 5,
    "dropout": 0.1,
    "top_k": 10,
    "min_similarity": None,
    "graph_temperature": 0.1,
    "cls_temperature": 0.1,
    "initial_patch_residual_scale": 0.05,
    "initial_score_residual_scale": 0.05,
    "patch_support_reduction": "mean_image",
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "graph_microbatch_size": 2,
    "train_queries_per_class": 1,
    "eval_queries_per_class": 15,
    "train_seed": 123,
    "val_seed": 456,
    "train_num_episodes": 1000,
    "val_num_episodes": 600,
    "num_epochs": 10,
    "train_episodes_per_epoch": 100,
    "validation_episodes_per_epoch": 20,
    "final_validation_episodes": 100,
    "early_stopping_patience": 3,
    "max_cached_shards": 6,
}
print(json.dumps(run_config, indent=2))


## Restore train and validation caches

From this point onward, tune architecture only on validation. The earlier test run should be treated as exploratory; do not repeatedly choose hyperparameters based on test accuracy.


In [ ]:
restore_feature_splits(
    ["train", "val"],
    paths.drive_feature_dir,
    paths.local_feature_dir,
)

train_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir, "train", max_cached_shards=run_config["max_cached_shards"]
)
val_features = MiniImageNetFeatureDataset(
    paths.local_feature_dir, "val", max_cached_shards=run_config["max_cached_shards"]
)

train_episodes = FewShotFeatureEpisodeDataset(
    train_features,
    run_config["n_way"],
    run_config["k_shot"],
    run_config["train_queries_per_class"],
    num_episodes=run_config["train_num_episodes"],
    seed=run_config["train_seed"],
    vary_by_epoch=True,
)

val_episodes = FewShotFeatureEpisodeDataset(
    val_features,
    run_config["n_way"],
    run_config["k_shot"],
    run_config["eval_queries_per_class"],
    num_episodes=run_config["val_num_episodes"],
    seed=run_config["val_seed"],
    vary_by_epoch=False,
)

graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(train_features.metadata["grid_size"]),
    top_k=run_config["top_k"],
    min_similarity=run_config["min_similarity"],
    graph_dtype=torch.float32,
    similarity_device=device,
)


## Check 1 — frozen local patch matching

This isolates the core local-matching idea from message passing.

If frozen local patch matching beats frozen mean-patch pooling, local correspondences contain useful signal and the main weakness is likely the current GNN/readout. If it does not, then the local correspondence signal itself is weak on this benchmark.


In [ ]:
@torch.inference_mode()
def eval_non_gnn_baselines(dataset, num_episodes=20):
    stats = {"CLS": [0, 0], "MeanPatch": [0, 0], "LocalPatch": [0, 0]}

    for i in range(num_episodes):
        episode = dataset[i]
        cls_logits, targets = frozen_baseline_episode(
            episode, "cls", device, temperature=run_config["cls_temperature"]
        )
        mean_logits, mean_targets = frozen_baseline_episode(
            episode, "mean_patch", device, temperature=run_config["graph_temperature"]
        )
        patch_logits, patch_targets = frozen_patch_match_episode(
            episode,
            device,
            temperature=run_config["graph_temperature"],
            support_reduction=run_config["patch_support_reduction"],
        )
        assert torch.equal(targets, mean_targets) and torch.equal(targets, patch_targets)

        for name, logits in [
            ("CLS", cls_logits),
            ("MeanPatch", mean_logits),
            ("LocalPatch", patch_logits),
        ]:
            pred = logits.argmax(dim=-1)
            stats[name][0] += int((pred == targets).sum())
            stats[name][1] += targets.numel()

    return {name: correct / total for name, (correct, total) in stats.items()}

baseline_smoke = eval_non_gnn_baselines(val_episodes, num_episodes=20)
print(baseline_smoke)


## Check 2 — inspect semantic-edge quality before pruning

Do not invent a similarity threshold. First inspect the selected semantic-edge cosine distribution on validation. If the lower tail is weak, then later compare `top_k ∈ {3,5,10}` and/or a threshold chosen on validation.


In [ ]:
def semantic_similarity_stats(builder, episode, max_graphs=20):
    values = []
    built = 0
    N = episode["support_patches"].shape[0]
    Q = episode["query_patches"].shape[1]

    for qc in range(N):
        for qp in range(Q):
            for candidate in range(N):
                graph = builder.build_graph(
                    query_patches=episode["query_patches"][qc, qp],
                    support_patches=episode["support_patches"][candidate],
                    candidate_id=candidate,
                )
                semantic = graph.edge_type.reshape(-1) == builder.SEMANTIC_EDGE
                values.append(graph.edge_attr[semantic, 0].float().cpu())
                built += 1
                if built >= max_graphs:
                    x = torch.cat(values)
                    q = torch.quantile(x, torch.tensor([0, .1, .25, .5, .75, .9, 1.0]))
                    return {
                        "mean": float(x.mean()),
                        "quantiles": [float(v) for v in q],
                        "num_semantic_edges": int(x.numel()),
                    }

edge_stats = semantic_similarity_stats(graph_builder, val_episodes[0])
print(json.dumps(edge_stats, indent=2))


## Build the improved model

The key tensor path is:

`387-D node feature → GATv2 256-D hidden → 384-D correction`

then

`refined_patch = original_DINO_384 + beta × correction_384`.

The readout no longer performs `mean(256 patches) → cosine`; it scores local query/support correspondences directly.


In [ ]:
base_gat = PatchGATv2Encoder(
    input_dim=run_config["input_dim"],
    hidden_dim=run_config["hidden_dim"],
    num_layers=run_config["num_layers"],
    heads=run_config["attention_heads"],
    edge_dim=run_config["edge_dim"],
    dropout=run_config["dropout"],
)

encoder = DinoResidualGraphEncoder(
    base_encoder=base_gat,
    base_output_dim=run_config["hidden_dim"],
    dino_dim=run_config["dino_dim"],
    initial_patch_residual_scale=run_config["initial_patch_residual_scale"],
)

readout = MaxMeanPatchReadout(
    temperature=run_config["graph_temperature"],
    support_reduction=run_config["patch_support_reduction"],
)

graph_matcher = CrossImageGraphMatcher(encoder=encoder, readout=readout)
model = BaselinePreservingResidualMatcher(
    graph_matcher,
    run_config["initial_score_residual_scale"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=run_config["learning_rate"],
    weight_decay=run_config["weight_decay"],
)

print("Initial alpha:", float(model.residual_scale.detach().cpu()))
print("Initial beta:", float(encoder.patch_residual_scale.detach().cpu()))


## Train with the existing residual routine

No new training loop is needed: the improved graph matcher still returns one score per candidate graph, so the existing episodic residual cross-entropy routine remains valid.


In [ ]:
checkpoint_dir = paths.drive_checkpoint_dir / run_config["experiment_name"]
result_dir = paths.drive_results_dir / run_config["experiment_name"]
checkpoint_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)
latest = checkpoint_dir / "latest.pt"
best = checkpoint_dir / "best.pt"

history = []
start_epoch = 0
best_accuracy = float("-inf")
without_improvement = 0
RESUME = True

if RESUME and latest.exists():
    state = load_training_checkpoint(latest, model, optimizer, device)
    start_epoch = state["epoch"] + 1
    best_accuracy = state["best_validation_accuracy"]
    without_improvement = state["epochs_without_improvement"]
    history = state.get("history", [])


In [ ]:
for epoch in range(start_epoch, run_config["num_epochs"]):
    print(f"\nEpoch {epoch + 1}/{run_config['num_epochs']}")

    train_metrics = train_residual_epoch(
        model, optimizer, graph_builder, train_episodes, device, epoch,
        run_config["train_episodes_per_epoch"],
        run_config["graph_microbatch_size"],
        run_config["cls_temperature"],
        log_interval=10,
    )

    val_metrics = evaluate_residual_dataset(
        model, graph_builder, val_episodes, device,
        run_config["validation_episodes_per_epoch"],
        run_config["graph_microbatch_size"],
        run_config["cls_temperature"],
        log_interval=5,
    )

    record = {
        "epoch": epoch,
        "train_loss": train_metrics.loss,
        "train_accuracy": train_metrics.accuracy,
        "validation_loss": val_metrics.loss,
        "validation_accuracy": val_metrics.accuracy,
        "score_alpha": float(model.residual_scale.detach().cpu()),
        "patch_beta": float(encoder.patch_residual_scale.detach().cpu()),
    }
    history.append(record)

    improved = val_metrics.accuracy > best_accuracy
    if improved:
        best_accuracy = val_metrics.accuracy
        without_improvement = 0
    else:
        without_improvement += 1

    checkpoint = make_checkpoint(
        model=model,
        optimizer=optimizer,
        epoch=epoch,
        best_validation_accuracy=best_accuracy,
        epochs_without_improvement=without_improvement,
        history=history,
        configuration=run_config,
    )
    save_checkpoint_atomic(checkpoint, latest)
    if improved:
        save_checkpoint_atomic(checkpoint, best)
    save_history(history, result_dir)

    print(record)

    if without_improvement >= run_config["early_stopping_patience"]:
        print("Early stopping.")
        break


## Final validation and diagnostics

The first question is not simply “did accuracy go up?” Check whether the model learns a useful patch correction (`beta`), whether the score residual (`alpha`) remains non-trivial, and whether the local-patch baseline itself is competitive.


In [ ]:
state = torch.load(best, map_location="cpu", weights_only=False)
model.load_state_dict(state["model_state_dict"])
model.to(device)
model.eval()

final_metrics = evaluate_residual_dataset(
    model, graph_builder, val_episodes, device,
    run_config["final_validation_episodes"],
    run_config["graph_microbatch_size"],
    run_config["cls_temperature"],
    log_interval=10,
)

result = {
    **final_metrics.to_dict(),
    "score_alpha": float(model.residual_scale.detach().cpu()),
    "patch_beta": float(encoder.patch_residual_scale.detach().cpu()),
    "baseline_smoke": baseline_smoke,
    "semantic_edge_stats": edge_stats,
}
atomic_json_save(result, result_dir / "validation_metrics.json")
print(json.dumps(result, indent=2))


## What to run next, depending on the result

- If **frozen local-patch > frozen mean-patch**, local correspondences are useful; focus on readout/GNN design.
- If **improved residual GAT > old residual GAT on validation**, preserve this architecture and then tune `top_k`.
- If `beta` stays near zero, the GNN is not learning useful patch corrections; inspect edge quality and reduce graph noise before adding more capacity.
- If `alpha` stays small but the model fixes low-margin CLS mistakes, a **query-dependent confidence gate** becomes the next experiment.
- If selected semantic-edge lower quantiles are weak, test `top_k=3,5,10`; only then test a validation-chosen `min_similarity`.
